# CP5 - Operaciones Avanzadas I

### Como siempre, antes de empezar...

Si has cerrado el proyecto o has abierto `Nueva Ventana` en VS Code, pulsa en el primer icono del menú de la izquierda `Explorador` y después en `Abrir Carpeta`. 

Busca la ruta en la que guardas el repositorio de GitHub en tu PC, normalmente *`Documentos => GitHub => fundamentos_big_data` * y pulsa en `Abrir`. De esta forma, ya tendrás la carpeta de tu repositorio cargada en VS Code.

> ¡Importante! Antes de abrir esa carpeta, debes ir a GitHub Desktop y realizar un `pull` para asegurarte de que tus archivos están actualizados. También puedes hacer el pull desde VS Code.  

### Requisitos para arrancar el entonro
- Iniciar Docker (abrir docker desktop).
- Iniciar GitHub (abrir github desktop) y estar logueado.
- En el terminal, ejecutar el siguiente comando en la raíz del proyecto `fundamentos_big_data %` 
```bash
docker-compose up --build
```
> ¡NO TE OLVIDES! Docker debe estar abierto y en ejecución en tu pc para poder utilizarlo. 


### Seleccionar Kernel
- Seleccionar kernel
- Servidor de Juypter existente
- Escriba la dirección URL del servidor de Jupyter en ejecución
- Seleccionar http://127.0.0.1:8888/?token=xxxxxxxx y pulsar Enter
- 127.0.0.1 y pulsar Enter
- Seleccionar Python 3 (ipykernel)

___________________

## Objetivos de la sesión

- Funciones de agregación con agrupación.
- Integrar datos mediante el uso de joins.
- Conocer y aplicar funciones de fecha y hora.

In [ ]:
### Importar las librerías necesarias ###
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

### INICIA LA SPARK-SESSION ###
spark = SparkSession.builder.appName("CP5").getOrCreate()

### Añade el spark context ###
sc = spark.sparkContext

### Añade el path para acceder a los datos y nómbralo como DATA_PATH ###
DATA_PATH = "/home/jovyan/data/"

### Lee la carpeta vuelos con pyspark y nómbralo como df_v ###
df_v = spark.read.parquet(DATA_PATH + "vuelos")

> **Importante**: a lo largo del notebook hay código que está mal o no funciona. Tendrás que modificarlo para que funcione. Por ejemplo, si cuando importes las funciones de pyspark como F, deberás añadir `F.` a las funciones que lo necesiten, como `col` o `desc`.

## 2 - Dataframes en PySpark

### 2.10 Agregación con Agrupación

El primer paso es agrupar usando la transformación `groupBy(<col1>,<col2>,...)`, especificando por qué columnas agrupamos las filas. El segundo paso es añadir las funciones de agregación deseadas.

#### **Función `groupBy()`**

Agrupa las filas en función de las columnas seleccionadas. Está supeditada a que se le aplique una función de agregación como `count()`,`min()`,`max()`, etc.

In [ ]:
(df_v.groupBy('ORIGIN_AIRPORT')
    .count()
    .orderBy(F.desc('count'))
).show(10)

In [ ]:
(df_v.groupBy('ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')
    .count()
    .orderBy(F.desc('count'))
).show(10)

#### **Función `agg()`**

Nos permite realizar **múltiples agregaciones** por grupo en lugar de una.

In [ ]:
df_v.groupBy('ORIGIN_AIRPORT').agg(
    F.count('AIR_TIME').alias('tiempo_aire'),
    F.min('AIR_TIME').alias('minimo'),
    F.max('AIR_TIME').alias('maximo')
).orderBy(F.desc('tiempo_aire')).show(10)

In [ ]:
df_v.groupBy('MONTH').agg(
    F.count('ARRIVAL_DELAY').alias('conteo_retrasos'),
    F.avg('DISTANCE').alias('promedio_dist')
).orderBy(F.desc('conteo_retrasos')).show(10)

#### **Función `pivot()`**

Nos permite realizar una agregación **pivotando las filas por las columnas**. Como parámetros se le puede añadir la columna sobre la que se pivota y entre corchetes `[]` podemos poner alguna de las categorías de esa variable.

In [ ]:
df_e = spark.read.parquet(DATA_PATH + 'estudiantes.parquet')

In [ ]:
df_e.show()

In [ ]:
df_e.groupBy('graduacion').pivot('sexo').agg(
    F.avg('peso')
).show(5)

In [ ]:
df_e.groupBy('graduacion').pivot('sexo', ['M']).agg(
    F.avg('peso')
).show(5)

### 2.11 Joins

Se trata de una técnica de combinación de tablas usando una columna común entre ellas para obtener como resultado un dataset que es combinación de ellos. Existen distintos tipos de Joins:

- **Inner Join** (por defecto en Spark): elementos comunes entre A y B.
- **Left Outer Join**: elementos de A y los compartidos con B.
- **Right Outer Join**: elementos de B y los compartidos con A.
- **Outer Join**: elementos de A, de B y de la combinación de ambos.
- **Left anti Join**: elementos de A que no están en B (diferencia).
- **Left Semi Join**: elementos de A y compartidos con B, pero solo muestra los de A.
- **Cross Join**: producto escalar de A y B.

La estructura fundamental de todos los métodos `join` es:

```python
df1.join(df2, <coldf1> == <coldf2>, '<join_type>')
```

Alternativamente, tenemos el `crossJoin`:

```python
df1.crossJoin(df2)
```

In [ ]:
emple = spark.read.parquet(DATA_PATH + 'empleados.parquet')
depart = spark.read.parquet(DATA_PATH + 'departamentos.parquet')

In [ ]:
emple.show()
depart.show()

#### Inner Join

In [ ]:
df8 = emple.join(depart, F.col('num_dpto') == F.col('id')) # por defecto es inner
df8.show()

In [ ]:
# Forma alternativa usando where()
emple.join(depart).where(F.col('num_dpto') == F.col('id')).show()

#### Left Outer Join

Se puede usar: `leftouter`, `left` o `left_outer`

In [ ]:
emple.join(depart, F.col('num_dpto') == F.col('id'), 'leftouter').show()

#### Right Outer Join

Se puede usar: `rightouter`, `right` o `right_outer`

In [ ]:
emple.join(depart, F.col('num_dpto') == F.col('id'), 'rightouter').show()

#### Outer Join

In [ ]:
emple.join(depart, F.col('num_dpto') == F.col('id'), 'outer').show()

#### Left Anti Join

Se puede usar: `left_anti` o `anti`

In [ ]:
emple.join(depart, F.col('num_dpto') == F.col('id'), 'anti').show()

Cambiando el orden de los dataframes, obtendremos los elementos que están en `depart` y no están en `emple`.

In [ ]:
depart.join(emple, F.col('num_dpto') == F.col('id'), 'anti').show()

#### Left Semi Join

Se puede usar: `left_semi` o `semi`

In [ ]:
emple.join(depart, F.col('num_dpto') == F.col('id'), 'semi').show()

#### Cross Join

In [ ]:
emple.crossJoin(depart).show()

> **Nota importante**: si encontramos dos columnas con el mismo nombre a la hora de hacer un `join`, es fundamental renombrar alguna de las columnas o hacer uso de la estructura `<df>[<'columna'>]`

### 2.12 Funciones de Fecha y Hora

In [ ]:
df_f = spark.read.parquet(DATA_PATH + 'convertir')

In [ ]:
df_f.printSchema()

In [ ]:
df_f.show(truncate=False)

#### **Funcion `to_date()` y `to_timestamp`**

Sirven para cambiar a tipo fecha y a tipo timestamp los string con valores de fecha, especificando la forma en la que se recibe ese patrón del string (ej: dd-MM-yyyy).

In [ ]:
df_fecha = df_f.select(
    F.to_date(F.col('date')).alias('fecha'),
    F.to_timestamp(F.col('timestamp')).alias('reg_tiempo'),
    F.to_date(F.col('date_str'), 'dd-MM-yyyy').alias('fecha_2'),
    F.to_timestamp(F.col('ts_str'), 'dd-MM-yyyy mm:ss').alias('reg_tiempo_2')
)
df_fecha.show(truncate=False)

In [ ]:
df_fecha.printSchema()

#### **Función `date_format()`**

Nos sirve para cambiar de un formato de fecha a otro, por ejemplo: la columna `fecha` está en formato `yyyy-MM-dd`. Con `date_format()` podemos cambiarlo a `dd-MM-yyyy`.

In [ ]:
df_fecha.select(
    F.date_format(F.col('fecha'), 'dd-MM-yyyy').alias('fecha dd-MM-yyyy')
).show()

In [ ]:
df_calc = spark.read.parquet(DATA_PATH + 'calculo.parquet')

In [ ]:
df_calc.show()

In [ ]:
df_calc.printSchema()

#### **Funciones `datediff()`, `months_between()`y `last_day()`**

Nos permite saber los días que pasan entre dos fechas, los meses que pasan entre dos fechas y el último día de alguna fecha, respectivamente (funcionan con strings).


In [ ]:
df_calc.select(
    F.datediff(F.col('fecha_salida'), F.col('fecha_ingreso')).alias('dias'),
    F.months_between(F.col('fecha_salida'), F.col('fecha_ingreso')).alias('meses'),
    F.last_day(F.col('fecha_salida')).alias('ultimo_dia')
).show()

#### **Funciones `date_add()` y `date_sub()`**

Nos permiten añadir y eliminar días a una fecha determinada (funcionan con strings).

In [ ]:
df_calc.select(
    F.col('nombre'), F.col('fecha_ingreso'),
    F.date_add(F.col('fecha_ingreso'), 14).alias('mas_14_dias'),
    F.date_sub(F.col('fecha_ingreso'), 1).alias('menos_1_dia')
).show()

#### **Funciones `year()`, `month()`, `dayofmonth()`, `dayofyear()`, `hour()`, `minute()` y `second()`**

Nos permite obtener esos valores en concreto, extrayéndolos de una fecha (funcionan con strings).

In [ ]:
df_calc.select(
    F.col('baja_sistema'),
    F.year('baja_sistema'),
    F.month('baja_sistema'),
    F.dayofmonth('baja_sistema'),
    F.dayofyear('baja_sistema'),
    F.hour('baja_sistema'),
    F.minute('baja_sistema'),
    F.second('baja_sistema')
).show()

### 2.13 Funciones para Strings

In [ ]:
df_str = spark.createDataFrame([(' Spark ',)], ['nombre'])
df_str.show()

> **Nota**: el valor 'Spark' de la columna "nombre", tiene un espacio en blanco a la izquierda y otro a la derecha.

#### **Funciones `ltrim()`, `rtrim()` y `trim()`**

Son el equivalente en python a `lstrip()`, `rstrip()` y `strip()`.

In [ ]:
df_str.select(
    F.ltrim('nombre').alias('ltrim'),
    F.rtrim('nombre').alias('rtrim'),
    F.trim('nombre').alias('trim')
).show()

#### **Funciones `lpad()` y `rpad()`**

Sirve para rellenar a la izquierda o a la derecha con caracteres alfanuméricos. Se le pasa como parámetros:

- Columna que queremos seleccionar.
- Número total de caracteres que tendrá tras la transformación.
- Caracter de relleno.

In [ ]:
df_str.select(
    F.trim('nombre').alias('trim')
).select(
    F.lpad(F.col('trim'), 8, '-').alias('lpad'),
    F.rpad(F.col('trim'), 8, '=').alias('rpad')
).show()

#### **Funciones `concat_ws()`, `lower()`, `upper()`, `initcap()` y `reverse()`**

La función `concat_ws` tiene dos parámetros:

- Caracter de separación.
- Columna a la que aplicarlo.

In [ ]:
df_str2 = spark.createDataFrame([('Spark','es','maravilloso')],
                                ['sujeto', 'verbo', 'adjetivo'])
df_str2.show()

In [ ]:
df_str2.select(
    F.concat_ws(' ', F.col('sujeto'),F.col('verbo'),F.col('adjetivo')).alias('frase'),
).select(
    F.col('frase'),
    F.lower(F.col('frase')).alias('minuscula'),
    F.upper(F.col('frase')).alias('mayuscula'),
    F.initcap(F.col('frase')).alias('capital'),
    F.reverse(F.col('frase')).alias('reverse')
).show()

#### **Función `regexp_replace()`**

La función tiene 3 parámetros principales:

- Columna a la que aplicaremos cambios.
- Término(s) que queremos sustituir.
- Término que se añadirá.

In [ ]:
df_str3 = spark.createDataFrame([('voy a casa a por mis llaves',)], ['frase'])

In [ ]:
df_str3.select(
    F.regexp_replace(F.col('frase'), 'voy|por', 'ir').alias('nueva_frase')
).show(truncate=False)

### 2.14 Funciones para trabajar con Colecciones

In [ ]:
df_c = spark.read.parquet(DATA_PATH + 'colecciones')

In [ ]:
df_c.show(truncate=False)

In [ ]:
df_c.printSchema()

En este caso, tenemos un Schema que nos muestra que una de las variables es de tipo `array` y que incluye una colección de elementos de tipo `str`.

#### **Función `size()`, `sort_array()` y `array_contains()`**

In [ ]:
df_c.select(
    F.size(F.col('tareas')).alias('size'),
    F.sort_array(F.col('tareas')).alias('array_ordenado'),
    F.array_contains(F.col('tareas'), 'buscar agua').alias('contiene...')
).show(truncate=False)

#### **Función `explode()`**

Nos permite "deconstruir" un array en filas de una columna seleccionada.

In [ ]:
df_c.select(
    F.col('dia'),
    F.explode(F.col('tareas')).alias('tareas')
).show()

### 2.15 Otras funciones

#### **Función `when()`**

La función `when()` sirve para establecer condiciones. Se le introducen estos parámetros:

- Columna con condición lógica.
- Salida de esa condición.

Se puede añadir un valor por defecto usando la función `otherwise()`.

In [ ]:
df_p = spark.createDataFrame([('Jose',1),('Julia',2),('Katia',1),
                              (None, 3),('Raul',3)], ['nombre','pago'])
df_p.show()

In [ ]:
df_p.select(
    F.col('nombre'),
    (F.when(F.col('pago') == 1, 'pagado')
    .when(F.col('pago') == 2, 'no pagado')
    .otherwise('sin iniciar')).alias('status')
).show()

#### **Funciones `coalesce()` y `lit()`**

La función `coalesce()` va a recibir `n` columnas y va a devolver el **primer valor que no sea nulo**. Por su parte, `lit()` nos permite añadir un literal. En el ejemplo, como tenemos un valor a `null`, lo vamos a detectar con `coalesce()` y luego renombrarlo usando `lit()`.

In [ ]:
df_p.select(
    F.coalesce(F.col('nombre'), F.lit('sin nombre')).alias('nombre')
).show()